# 01- Non-ICU Hospital Admission Cohort Construction


Short summary:
[01 - Non-ICU Hospital Admission Cohort Construction (SHORT)](../docs/01_cohort_SHORT.md)

## 0. Overview

This notebook constructs the base analytic cohort of non-ICU hospital admissions from the MIMIC-IV database.
The goal is to define a clean, admission-level cohort by excluding any hospital admissions associated with an ICU stay.

The entire-stay ICU-excluded cohort supplies the original exposure and association branch (02–04b). The additional [04c prediction-time evaluation](04c_prediction_time_evaluation.ipynb) instead starts from the adult `base_admissions` table and defines eligibility using information available by each prediction instant; later ICU transfers remain followed.
This notebook constructs and validates the adult base, race-grouped base and entire-stay ICU-excluded cohort tables; it does not define medication exposure or fit outcome models.

ICU admissions are excluded in a subsequent SQL-based preprocessing step ([02_exclude_icu_admissions.sql](../sql/02_exclude_icu_admissions.sql)).

## 1. Purpose

The purpose of this notebook is to create a well-defined non-ICU hospital admission cohort at the HADM level.

Specifically, this notebook:
- Excludes all hospital admissions with any recorded ICU stay
- Ensures one row per hospital admission (`subject_id`, `hadm_id`)
- Performs minimal demographic harmonization (race grouping) to ensure modeling stability
- Produces a stable cohort table for downstream exposure definition and outcome analyses
  
No exposure variables, outcome definitions, or statistical modeling are defined or estimated in this notebook.

## 2. Data Sources
- **MIMIC-IV v3.1** (BigQuery public dataset)
- Project: `mimic-iv-portfolio`

**Source tables (BigQuery public dataset):**
- `physionet-data.mimiciv_3_1_hosp.admissions`
- `physionet-data.mimiciv_3_1_hosp.patients`
- `physionet-data.mimiciv_3_1_icu.icustays`

**Derived datasets created in this notebook:**
- `mimic-iv-portfolio.nonicu_raas.base_admissions`<br>
  (created in 01_cohort.ipynb using [01_build_base_hosp_admissions.sql](../sql/01_build_base_hosp_admissions.sql)

- `mimic-iv-portfolio.nonicu_raas.base_admissions_racegrp`<br>
  (created in 01_cohort.ipynb using [01_collapse_race_categories.sql](../sql/01_collapse_race_categories.sql)

- `mimic-iv-portfolio.nonicu_raas.nonicu_admissions`<br>
  (created in 01_cohort.ipynb using [02_exclude_icu_admissions.sql](../sql/02_exclude_icu_admissions.sql)

## 3. Build Cohort and Exposure Tables in BigQuery
### 3.1 BigQuery Client Initialization and Helper Functions

## Reading the results

Data sources and execution mode are summarized once below, after the destination settings. Expand **Calculation inputs and source definitions** to trace individual SQL definitions and aggregate references.

Each result is generated by Python in this order: **number/title → figure or aggregate table → how to read → computed result summary → interpretation limits**. Cohort definitions, exposures, adjustment variables and estimators are unchanged. See the [reproduction instructions](../REPRODUCIBILITY.md#python-generated-notebook-results).


The input-definition index describes the Python objects/queries used in calculation. The separate **Saved aggregates and verification files (CSV/JSON)** list identifies saved results and checking records, rather than original MIMIC-IV source tables. Live notebook displays use in-memory results; saved-output replay and validation workflows may read these files.

In [ ]:
# Presentation uses only objects produced by this notebook execution.
from pathlib import Path as _ResultPath
import sys as _result_sys
_result_root = _ResultPath.cwd().resolve()
if _result_root.name == "notebooks":
    _result_root = _result_root.parent
if not (_result_root / "scripts" / "nonicu_notebook_results.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory")
if str(_result_root / "scripts") not in _result_sys.path:
    _result_sys.path.insert(0, str(_result_root / "scripts"))
from nonicu_notebook_results import NotebookResults
notebook_results = NotebookResults(globals())


In [ ]:
# Use Application Default Credentials (my user account)
# This account already has PhysioNet BigQuery access.

from google.cloud import bigquery
from google.auth import default
import pandas as pd
from pathlib import Path

from google.cloud import bigquery
from google.auth import default
from pathlib import Path

# 1. Define project ID, dataset, and table references
PROJECT_ID = "mimic-iv-portfolio"
DATASET = "nonicu_raas"

TABLE_BASE=f"{PROJECT_ID}.{DATASET}.base_admissions"         # Created in 01_build_base_hosp+admissions.sql
TABLE_BASE_RACE = f"{PROJECT_ID}.{DATASET}.base_admissions_racegrp"  # Created in 01_collapse_race_categories.sql
TABLE_NONICU = f"{PROJECT_ID}.{DATASET}.nonicu_admissions"   # Created in 02_exclude_icu_admissions.sql


# 2. Get ADC credentials
creds, adc_project = default()

# 3. Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID, credentials=creds)

print("Connected to BigQuery project:", PROJECT_ID)
print("ADC default project:", adc_project)

# 4. Helper to run a SQL script file (DDL, CREATE TABLE, etc.)
def run_sql_script(path) :
    """
    Read a .sql file from disk, execute it in BigQuery,
    and wait until the job finishes.
    Use this for CREATE TABLE / INSERT INTO scripts.
    """
    sql_path = Path(path)
    with sql_path.open("r") as f:
        query = f.read()
    job = client.query(query)
    job.result()
    print(f"Executed SQL script: {sql_path.name}")

In [ ]:
# Shared source references and execution mode for all numbered results.
notebook_results.overview(['Table 01.1', 'Table 01.2', 'Table 01.3', 'Table 01.4', 'Table 01.5', 'Table 01.6', 'Table 01.7', 'Table 01.8', 'Table 01.9', 'Table 01.10', 'Table 01.11', 'Table 01.12', 'Table 01.13'])


### 3.2 Build Base Adult Hospital Admissions (One Row per Admission)

This step constructs a base adult hospital-admission–level cohort from the MIMIC-IV v3.1 database, serving as the foundation for downstream non-ICU analyses.

- SQL script: [01_build_base_hosp_admissions.sql](../sql/01_build_base_hosp_admissions.sql)
- Output table: `nonicu_raas.base_admissions`

The SQL script extracts hospital admission records from the
`physionet-data.mimiciv_3_1_hosp.admissions` table and links them to patient-level demographic information from the `physionet-data.mimiciv_3_1_hosp.patients` table using `subject_id`.

Key admission-level variables include admission and discharge timestamps, in-hospital mortality indicators, admission and discharge locations, insurance status, language, marital status, and recorded race. Patient-level variables include sex and de-identified anchor variables derived
from the `physionet-data.mimiciv_3_1_hosp.patients` table.

An approximate age at admission is derived using the standard MIMIC-IV de-identification approach, calculated as the patient’s `anchor_age` plus the difference between the calendar year of hospital admission and the `anchor_year`. Hospital length of stay is calculated in days based on the difference between discharge time and admission time.

The final cohort is restricted to adult admissions (age ≥ 18 years) to ensure clinical relevance and consistency with downstream analyses. The resulting table provides a clean, admission-level dataset with harmonized demographic and temporal variables, which serves as a core input for cohort refinement, exposure definition, and outcome modeling in subsequent steps.

In [ ]:
# Excecute SQL script (01_build_base_hosp_admissions.sql)
run_sql_script("../sql/01_build_base_hosp_admissions.sql")

### 3.3 Load and Inspect Base Admissions Table

In [ ]:
with notebook_results.capture('Table 01.1', globals()):
    query_base = f"""
SELECT *
FROM `{TABLE_BASE}`
"""

    df_base = client.query(query_base).to_dataframe()

    df_base.info()
    print("Shape:", df_base.shape)
    _notebook_result_value = (df_base["age"].describe())


### 3.4 Construction of Collapsed Race Categories

This step derives a collapsed race variable to reduce category sparsity and improve model stability in downstream regression analyses.

- SQL script: [01_collapse_race_categories.sql](../sql/01_collapse_race_categories.sql)
- Output table: `nonicu_raas.base_admissions_racegrp`

The SQL script extracts hospital admission records from the `physionet-data.mimiciv_3_1_hosp.admissions` table and links them to patient-level demographic information from the `physionet-data.mimiciv_3_1_hosp.patients` table using `subject_id`.

Key admission-level variables include admission and discharge timestamps, in-hospital mortality indicators, admission and discharge locations, insurance status, language, marital status, and recorded race. Patient-level variables include sex and de-identified anchor variables (`anchor_age` and `anchor_year`) provided in the patients table.

In [ ]:
# Excecute SQL script (01_collapse_race_categories.sql)
run_sql_script("../sql/01_collapse_race_categories.sql")

### 3.5 Validation of Collapsed Race Categories

In [ ]:
with notebook_results.capture('Table 01.2', globals()):
    query_base_race = f"""
SELECT *
FROM `{TABLE_BASE_RACE}`
"""

    df_base_race = client.query(query_base_race).to_dataframe()

    df_base_race.info()
    print("Shape:", df_base_race.shape)
    _notebook_result_value = (df_base_race["race_group"].value_counts(dropna=False))


### 3.6 Exclude ICU Admissions to Define the Non-ICU Cohort

This step excludes hospital admissions that involved any ICU stay in order to construct a non-ICU adult inpatient cohort suitable for downstream analyses focused on general ward populations.

- SQL script: [02_exclude_icu_admissions.sql](../sql/02_exclude_icu_admissions.sql)
- Output table: `nonicu_raas.nonicu_admissions`

The SQL script identifies hospital admissions associated with at least one ICU stay using the `physionet-data.mimiciv_3_1_icu.icustays` table, which records individual ICU stay episodes. Distinct combinations of `subject_id` and `hadm_id` are used to represent admissions with any ICU exposure.

These ICU-associated admissions are then excluded from the base admission cohort stored in `nonicu_raas.base_admissions_racegrp` using a left join and null-filtering strategy. Admissions that do not match any ICU stay records are retained.

The resulting table contains adult hospital admissions without any ICU stay during the hospitalization, preserving all admission-level and patient-level variables carried forward from the base cohort. This restriction ensures clear separation between ICU and non-ICU populations and supports analyses specifically targeting outcomes among general (non-ICU) inpatient admissions.

In [ ]:
# Excecute SQL script (02_exclude_icu_admissions.sql)
run_sql_script("../sql/02_exclude_icu_admissions.sql")

### 3.7 Load and Inspect the Non-ICU Admissions Table

In [ ]:
with notebook_results.capture('Table 01.3', globals()):
    query_nonicu = f"""
SELECT *
FROM `{TABLE_NONICU}`
"""
    df_nonicu = client.query(query_nonicu).to_dataframe()

    df_nonicu.info()
    print("Shape:", df_nonicu.shape)
    _notebook_result_value = (df_nonicu["race_group"].value_counts(dropna=False))


## 4. Sanity Checks

### 4.1 Sanity Check: Non-ICU Cohort Proportion

In [ ]:
with notebook_results.capture('Table 01.4', globals()):
    cohort_summary_query = f"""
WITH base AS (
  SELECT COUNT(*) AS n_base
  FROM `{TABLE_BASE}`
),
nonicu AS (
  SELECT COUNT(*) AS n_nonicu
  FROM `{TABLE_NONICU}`
)
SELECT
  n_base,
  n_base - n_nonicu AS n_excluded_icu,
  n_base AS n_adult_admissions,
  n_nonicu
FROM base, nonicu
"""

    _notebook_result_value = (client.query(cohort_summary_query).to_dataframe())


### 4.2 Base cohort descriptive statistics (sanity checks)

In [ ]:
with notebook_results.capture('Table 01.5', globals()):
    summary_query = f"""
WITH stats AS (
  SELECT
    AVG(age) AS age_mean,
    APPROX_QUANTILES(age, 4)[OFFSET(1)] AS age_q1,
    APPROX_QUANTILES(age, 2)[OFFSET(1)] AS age_median,
    APPROX_QUANTILES(age, 4)[OFFSET(3)] AS age_q3,
    AVG(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) AS female_fraction,
    AVG(CASE WHEN gender = 'M' THEN 1 ELSE 0 END) AS male_fraction,
    AVG(CAST(hospital_expire_flag AS FLOAT64)) AS crude_mortality_rate
  FROM `{TABLE_NONICU}`
)
SELECT 'Age (mean)' AS metric, CAST(age_mean AS STRING) AS value FROM stats
UNION ALL SELECT 'Age (median)', CAST(age_median AS STRING) FROM stats
UNION ALL SELECT 'Age (IQR)', CONCAT(CAST(age_q1 AS STRING), '-', CAST(age_q3 AS STRING)) FROM stats
UNION ALL SELECT 'Female fraction', CAST(female_fraction AS STRING) FROM stats
UNION ALL SELECT 'Male fraction', CAST(male_fraction AS STRING) FROM stats
UNION ALL SELECT 'Crude in-hospital mortality rate', CAST(crude_mortality_rate AS STRING) FROM stats
"""

    _notebook_result_value = (client.query(summary_query).to_dataframe())


**Missingness note**

Several baseline variables (e.g., insurance, language, marital status) contain partial missing values.
In downstream modeling steps, missingness will be handled explicitly by including a "Missing" category
for categorical covariates, rather than excluding observations.


### 4.3 Missingness Check for race_group

In [ ]:
with notebook_results.capture('Table 01.6', globals()):
    race_group_missing_query = f"""
SELECT COUNTIF(race_group IS NULL) AS n_missing_race_group
FROM `{TABLE_BASE_RACE}`
"""

    _notebook_result_value = (client.query(race_group_missing_query).to_dataframe())


### 4.4 Distribution of Collapsed Race Categories

In [ ]:
with notebook_results.capture('Table 01.7', globals()):
    race_group_counts_query = f"""
SELECT
  race_group,
  COUNT(*) AS n
FROM `{TABLE_BASE_RACE}`
GROUP BY race_group
ORDER BY n DESC, race_group
"""

    _notebook_result_value = (client.query(race_group_counts_query).to_dataframe())


### 4.5 Age Distribution in the Non-ICU Cohort

In [ ]:
with notebook_results.capture('Table 01.8', globals()):
    age_distribution_query = f"""
SELECT
  COUNT(age) AS n_nonmissing,
  AVG(age) AS mean,
  STDDEV_SAMP(age) AS std,
  MIN(age) AS min,
  APPROX_QUANTILES(age, 4)[OFFSET(1)] AS q1,
  APPROX_QUANTILES(age, 2)[OFFSET(1)] AS median,
  APPROX_QUANTILES(age, 4)[OFFSET(3)] AS q3,
  MAX(age) AS max
FROM `{TABLE_NONICU}`
"""

    _notebook_result_value = (client.query(age_distribution_query).to_dataframe())


### 4.6 In-Hospital Mortality Indicator Distribution

In [ ]:
with notebook_results.capture('Table 01.9', globals()):
    mortality_counts_query = f"""
SELECT
  hospital_expire_flag,
  COUNT(*) AS n
FROM `{TABLE_NONICU}`
GROUP BY hospital_expire_flag
ORDER BY hospital_expire_flag
"""

    _notebook_result_value = (client.query(mortality_counts_query).to_dataframe())


### 4.7 Hospital Length of Stay Distribution

In [ ]:
with notebook_results.capture('Table 01.10', globals()):
    los_distribution_query = f"""
SELECT
  COUNT(hosp_los) AS n_nonmissing,
  AVG(hosp_los) AS mean,
  STDDEV_SAMP(hosp_los) AS std,
  MIN(hosp_los) AS min,
  APPROX_QUANTILES(hosp_los, 4)[OFFSET(1)] AS q1,
  APPROX_QUANTILES(hosp_los, 2)[OFFSET(1)] AS median,
  APPROX_QUANTILES(hosp_los, 4)[OFFSET(3)] AS q3,
  MAX(hosp_los) AS max
FROM `{TABLE_NONICU}`
"""

    _notebook_result_value = (client.query(los_distribution_query).to_dataframe())


### 4.7 Distribution of Hospital Length of Stay (LOS)

In [ ]:
with notebook_results.capture('Table 01.11', globals()):
    los_histogram_query = f"""
SELECT
  CAST(FLOOR(hosp_los) AS INT64) AS hosp_los_day_bin,
  COUNT(*) AS n
FROM `{TABLE_NONICU}`
WHERE hosp_los IS NOT NULL
GROUP BY hosp_los_day_bin
ORDER BY hosp_los_day_bin
"""

    _notebook_result_value = (client.query(los_histogram_query).to_dataframe())


### 4.8 Verify Non-ICU Cohort Has Zero ICU Admissions

In [ ]:
with notebook_results.capture('Table 01.12', globals()):
    # This should always be zero
    query_icu_check = f"""
SELECT COUNT(*) AS n_icu
FROM `{TABLE_NONICU}` n
JOIN `physionet-data.mimiciv_3_1_icu.icustays` i
  ON n.subject_id = i.subject_id
 AND n.hadm_id   = i.hadm_id
"""
    _notebook_result_value = (client.query(query_icu_check).to_dataframe())


### 4.9 Cohort Size and Key Demographic Summary

In [ ]:
with notebook_results.capture('Table 01.13', globals()):
    final_summary_query = f"""
WITH base AS (
  SELECT COUNT(*) AS n_base_admissions
  FROM `{TABLE_BASE}`
),
nonicu AS (
  SELECT
    COUNT(*) AS n_nonicu_admissions,
    AVG(CAST(hospital_expire_flag AS FLOAT64)) * 100 AS mortality_rate_percent,
    APPROX_QUANTILES(age, 2)[OFFSET(1)] AS median_age
  FROM `{TABLE_NONICU}`
)
SELECT
  n_base_admissions,
  n_nonicu_admissions,
  mortality_rate_percent,
  median_age
FROM base, nonicu
"""

    _notebook_result_value = (client.query(final_summary_query).to_dataframe())


## 5. Outputs and Downstream Use

This notebook constructs foundational hospital admission–level cohort tables in BigQuery that supply two distinct downstream branches: the entire-stay non-ICU association analyses and the separate time-specific prediction evaluation.

**Outputs**

As a result of executing the SQL scripts in this notebook, the following tables are materialized in the `mimic-iv-portfolio.nonicu_raas` dataset:

- **base_admissions**
An adult hospital admissions table derived from MIMIC-IV v3.1, containing one row per hospital admission.
This table includes admission timing, demographic variables, insurance and admission metadata, an approximate age at admission derived from anchor variables, and hospital length of stay. It serves as the upstream input for race category collapsing and subsequent cohort refinement steps, including [01_collapse_race_categories.sql](../sql/01_collapse_race_categories.sql).

- **base_admissions_racegrp**
An intermediate derived table created by collapsing raw race categories into predefined race groups (e.g., White, Black, Hispanic, Asian, Other).
This table is used as an input to the ICU-exclusion step [02_exclude_icu_admissions.sql](../sql/02_exclude_icu_admissions.sql) to ensure consistent race categorization prior to construction of the final non-ICU cohort.

- **nonicu_admissions** 
A refined admission-level cohort created by excluding all hospital admissions with any recorded ICU stay.
Each row represents a single adult, non-ICU hospital admission, uniquely identified by (`subject_id`, `hadm_id`).

These tables are constructed entirely via SQL and represent finalized cohort-selection outputs of the data preparation pipeline.

**Downstream Use**

The nonicu_admissions table serves as the core cohort input for downstream notebooks, including:

- Early RAAS inhibitor exposure definition ([02_exposure.ipynb](02_exposure.ipynb))<br>
- Construction of the unified analysis dataset in [02_exposure.ipynb](02_exposure.ipynb), followed by read-only input checks in [03a_validate_input_tables.ipynb](03a_validate_input_tables.ipynb)<br>
- Baseline characteristic summaries ([03b_describe_analysis_dataset.ipynb](03b_describe_analysis_dataset.ipynb))<br>
- Unadjusted outcome summaries and crude analyses ([04a_unadjusted_outcomes.ipynb](04a_unadjusted_outcomes.ipynb))<br>
- Multivariable and adjusted outcome modeling with demographic covariates ([04b_multivariable_outcomes.ipynb](04b_multivariable_outcomes.ipynb))<br>

Notebook 02 constructs exposure and analysis tables. Notebooks 03a–04b read the constructed tables without rewriting them; 04b additionally applies its documented landmark eligibility filters.
This design ensures a clear separation between cohort construction, exposure definition, and analytic modeling, supporting reproducibility and transparent data provenance.

Together, these tables define a transparent, stepwise cohort construction pipeline that separates raw data ingestion, demographic preprocessing, and cohort restriction.

---
## Next Step

Proceed to:
- [02. Build Non-ICU RAAS Early Exposure Tables](02_exposure.ipynb)

This next notebook/document briefly describes the next workflow step.

**Separate prediction branch:** [04c](04c_prediction_time_evaluation.ipynb) does not reuse `nonicu_admissions` to select its risk sets. It starts from `base_admissions`, confirms ward location and no ICU entry up to admission/24 hours, and retains later ICU transfers through discharge. Its cohorts and metrics must be kept separate from the original association cohort. See [current execution evidence](../docs/CLINICAL_REEXECUTION_615de01.md).
